# Chapter 4 — Model Context Protocol (MCP) · Homework (self-test)

**Goal:** check that you can choose the right MCP primitive, transport, scope and error contract, and that you can write the code that connects MCP servers to Claude.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 1.5–2 hours
- Estimated API cost: $0 (everything is graded offline); the optional live bonus costs < $0.01 with the default model.
- Prerequisites: finish the practice notebook; repo venv from [00-prerequisites](../../00-prerequisites/README.md).

**How to use it**

1. Work top to bottom. Attempt every question and exercise **before** opening the solutions at the very bottom.
2. **Part A** — 12 scenario questions. Put your letters in the `answers` dict and run the grader.
3. **Part B** — 8 coding exercises. Replace each `TODO` stub, then run its `check_exN()` cell. Checkers never crash the notebook; they print ✓ or a hint.
4. **Part C** — one architecture scenario. Write your answer, then compare with the rubric.
5. **Scorecard** — your overall score is the average of the quiz % and the exercise %. The pass mark is **72%**, mirroring the Claude certification exam's 720/1000.

In [ ]:
import asyncio
import base64
import hashlib
import json
import os
import re
from types import SimpleNamespace as NS
from typing import Any, Awaitable, Callable

import anthropic
from dotenv import find_dotenv, load_dotenv
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from mcp.types import (BlobResourceContents, CallToolResult, Completion, CompletionArgument, CompletionContext,
                       ImageContent, PromptReference, ResourceTemplateReference, TextContent, Tool)

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
client = anthropic.Anthropic()        # only the optional live bonus uses Claude
aclient = anthropic.AsyncAnthropic()

RESULTS: dict[str, Any] = {"quiz": None, "exercises": {}}
N_EXERCISES = 8


def _record(n: int, ok: bool, msg: str = "") -> None:
    RESULTS["exercises"][n] = ok
    print(f"✓ Exercise {n} passed" if ok else f"✗ Exercise {n}: {msg}")


def run_check(n: int, fn: Callable[[], None]) -> None:
    try:
        fn()
        _record(n, True)
    except NotImplementedError:
        _record(n, False, "not implemented yet (replace the TODO stub).")
    except AssertionError as e:
        _record(n, False, str(e) or "an assertion failed.")
    except Exception as e:  # noqa: BLE001 - graders must never crash the notebook
        _record(n, False, f"your code raised {type(e).__name__}: {e}")


async def arun_check(n: int, fn: Callable[[], Awaitable[None]]) -> None:
    try:
        await fn()
        _record(n, True)
    except NotImplementedError:
        _record(n, False, "not implemented yet (replace the TODO stub).")
    except AssertionError as e:
        _record(n, False, str(e) or "an assertion failed.")
    except BaseException as e:  # noqa: BLE001 - includes ExceptionGroup from anyio task groups
        while isinstance(e, BaseExceptionGroup) and len(e.exceptions) == 1:
            e = e.exceptions[0]
        if isinstance(e, NotImplementedError):
            _record(n, False, "not implemented yet (replace the TODO stub).")
        elif isinstance(e, AssertionError):
            _record(n, False, str(e) or "an assertion failed.")
        else:
            _record(n, False, f"your code raised {type(e).__name__}: {e}")

In [ ]:
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites). Only the optional live bonus needs it."

---
## Part A — Quiz (12 questions)

Choose one letter per question. Each question has exactly one best answer.

**Q1.** A platform team wants on-call engineers using Claude Code to start a standard "summarize incident" workflow by picking it from a menu and typing an incident ID. The workflow wording should live, versioned, on the team's MCP server. Which MCP primitive fits?

- A) A tool `summarize_incident`, so Claude decides when to run it.
- B) A resource template `incidents://{id}` that the host attaches automatically.
- C) A prompt `summarize_incident(incident_id)`, which Claude Code surfaces as a slash command.
- D) A `read_only_hint=True` annotation on the existing incident tools.

**Q2.** An internal pricing MCP server runs today as a stdio process on each developer's laptop. Next quarter it must serve 40 developers in Claude Code **and** a production Messages API backend that uses the MCP connector. What is the target design?

- A) Keep stdio: the connector can launch the server if you pass its command in `mcp_servers`.
- B) Deploy one Streamable HTTP server behind TLS with OAuth or bearer-token auth; developers and the connector use the same URL.
- C) Deploy it with HTTP+SSE, since the connector lists SSE among supported transports.
- D) Keep stdio for developers and let the backend reach a developer laptop over an SSH tunnel.

**Q3.** A custom Python stdio server passes its unit tests, but in Claude Code it intermittently shows as failed and the debug log mentions JSON parse errors. The server prints a startup banner and progress messages with `print()`. What is the most likely fix?

- A) Raise `MAX_MCP_OUTPUT_TOKENS` so long messages are not truncated.
- B) Move the server from project scope to user scope.
- C) Make every tool return a `CallToolResult` with `is_error` set.
- D) Send all logging to stderr (the `logging` module or `print(..., file=sys.stderr)`): stdout is reserved for protocol messages.

**Q4.** A pull request adds a `.mcp.json` containing `"Authorization": "Bearer ghp_9f3..."` with the comment "replace with your own token". Six developers each have a personal GitHub token. What is the best review response?

- A) Approve: the repository is private, so the token is safe.
- B) Request changes: use `"Bearer ${GITHUB_PAT}"` expansion, document the variable in the README, and revoke the committed token.
- C) Ask every developer to add the server at user scope instead, and delete `.mcp.json`.
- D) Ask the author to write a proxy MCP server that reads tokens from a local `.env` file.

**Q5.** In one repository a developer has a `github` server at **user** scope (broad toolset), the committed `.mcp.json` defines `github` (team URL with `${GITHUB_PAT}`), and she also added `github` at **local** scope pointing to a GitHub Enterprise URL without headers. What does Claude Code use in this repository?

- A) The local-scope entry, used whole: no headers from the other definitions are merged in.
- B) The project entry, because committed configuration always wins.
- C) A merge: the local URL plus the project's `Authorization` header.
- D) The user-scope entry, because it was added first.

**Q6.** An inventory tool `find_sku` returns `isError: true` with the text "Not found" when an SKU does not exist. The agent retries several times and then tells customers that "the inventory system is down". What is the best fix?

- A) Add "never retry find_sku" to the system prompt.
- B) Raise `MCPError` so the failure is handled at the protocol level.
- C) Return a successful result with an explicit empty payload (for example `found: false`) and keep `isError` for real failures.
- D) Also return an empty result for timeouts, so the agent never sees errors.

**Q7.** After upgrading a server from MCP Python SDK v1 to v2, tools that `raise ValueError("email must contain '@'")` now make Claude see only `Error executing tool find_customer`, and it keeps guessing new arguments. What is the best fix?

- A) Pin `mcp<2` permanently.
- B) Raise `MCPError` with the message so the client receives it.
- C) Catch the `ValueError` and return an empty list.
- D) Raise `ToolError` with the message for anticipated failures (or return a `CallToolResult` with `is_error=True` and a structured payload).

**Q8.** A Messages API backend uses the MCP connector to reach a remote documentation server. The team expects Claude to read `docs://runbooks/{id}` resources and to use the server's `triage` prompt, but neither works. What is the most likely cause and fix?

- A) The connector supports tools only: expose runbook lookup as a read-only tool, or run your own MCP client bridge if resources and prompts are required.
- B) The beta header is outdated: switching to `mcp-client-2026-09-15` enables resources and prompts.
- C) Resources must be switched on in the toolset's `default_config`.
- D) The server must be converted to stdio for resources to work.

**Q9.** A developer-productivity agent connects 6 MCP servers (about 70 tools), but its role needs 5 of them. It often picks a similar-sounding wrong tool. What is the best first step?

- A) Switch to a larger model with a longer context window.
- B) Merge all tools into a single `do_anything(action, args)` tool.
- C) Connect only the servers the role needs and restrict it to the needed tools with `allowed_tools` (`mcp__<server>__<tool>` names).
- D) Add a system prompt that lists all 70 tools with usage rules.

**Q10.** A `delete_customer` tool is annotated `destructive_hint=True`. The security reviewer asks how you guarantee that deleting more than 10 customers per hour requires human approval. What is the correct answer?

- A) `destructive_hint` makes every MCP client require approval, so nothing more is needed.
- B) Enforce the limit deterministically in the server (or in a host-side hook); annotations are untrusted hints for UIs and permission prompts.
- C) Set `read_only_hint=False` as well; the protocol enforces the combination.
- D) Write the limit into the tool description so Claude respects it.

**Q11.** A chat app reads an MCP resource `stats://catalog` whose handler returns a Python `dict`. The app parses a read as JSON only when its `mime_type` is `application/json`, so users see a raw JSON string. The server author says "the SDK already returns JSON". What is the best fix?

- A) Return `bytes` from the handler so the client receives a blob.
- B) Turn the resource into a tool so Claude can parse the output itself.
- C) Change it into a resource template so the client can request `?format=json`.
- D) Declare `mime_type="application/json"` on the resource: the SDK serializes the dict to JSON text but labels it `text/plain` unless you say otherwise.

**Q12.** A custom Python host launches a stdio CRM server with `StdioServerParameters(command="python", args=["crm_server.py"])`. `CRM_TOKEN` is exported in the shell that runs the host, yet the server fails with "CRM_TOKEN not set". What is the right fix?

- A) Pass it explicitly, `env={"CRM_TOKEN": os.environ["CRM_TOKEN"]}`: the SDK's stdio client gives the child process only a small safe set of variables plus what you put in `env`.
- B) Hard-code the token in `crm_server.py`, since stdio servers are local anyway.
- C) Switch to Streamable HTTP, because stdio servers cannot receive credentials.
- D) Have the server print a token request to stdout at startup so the client can answer it.

In [ ]:
answers = {
    "q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
    "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
    "q11": None, "q12": None,
}

In [ ]:
ANSWER_KEY = {
    "q1": "227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e",
    "q2": "61e5302385b2dd8204dbddfd86b357cacb0de07dbc800460aa335d9bc96b6ca1",
    "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
    "q4": "61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa",
    "q5": "efeafc86344da0b52bbaa7783b6621595e70274062976345ce6c533b9322877c",
    "q6": "18cb69e7d9c041e4c1b21fa8c0e110716664b55bef1348d49301f8891c82cef1",
    "q7": "45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457",
    "q8": "f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc",
    "q9": "e19608e84d37f3c94b5f88738c75a8c96772a5e61784e3e24b5512fa65ff63c7",
    "q10": "c69a9867de9de4cc2e45f4cd6a6128833af714d8993edbeac622795c9c5e20d3",
    "q11": "19b8f64da90dd1e288d2fa60f4128e5e8c665ace09e8470b263486752199e135",
    "q12": "9d4e72c9d2036d777ce76bca900a85c706c8a653c712a6faf259d63490f850a2"
}  # sha256(f"{qid}:{letter}") so the letters are not visible


def grade_quiz(answers: dict) -> float:
    correct = 0
    for qid, digest in ANSWER_KEY.items():
        given = str(answers.get(qid) or "").strip().upper()[:1]
        ok = bool(given) and hashlib.sha256(f"{qid}:{given}".encode()).hexdigest() == digest
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {given or '(unanswered)'}")
    score = correct / len(ANSWER_KEY)
    RESULTS["quiz"] = score
    print(f"\nQuiz: {correct}/{len(ANSWER_KEY)} = {score:.0%}")
    return score


grade_quiz(answers)

---
## Part B — Coding exercises

All exercises run offline. Types you will use: `mcp.types.Tool` (`.name`, `.description`, `.input_schema`), `CallToolResult` (`.content`, `.structured_content`, `.is_error`), `TextContent(type="text", text=...)`.

### Exercise 1 — Namespaced Claude tool definitions

A host that bridges several MCP servers into one Messages API call must avoid name collisions. Write `to_claude_tool(server_name, tool)` that returns a Messages API tool definition with exactly the keys `name`, `description`, `input_schema`:

- `name` is `mcp__<server_name>__<tool.name>` (the Claude Code convention),
- `description` is the tool's description, or `""` if it has none,
- `input_schema` is the tool's JSON schema, unchanged.

In [ ]:
def to_claude_tool(server_name: str, tool: Tool) -> dict:
    # TODO: build the Messages API tool definition
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    schema = {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}
    t = Tool(name="lookup_order", description="Fetch one order.", input_schema=schema)
    out = to_claude_tool("orders", t)
    assert isinstance(out, dict), "return a dict"
    assert set(out) == {"name", "description", "input_schema"}, f"keys must be name/description/input_schema, got {sorted(out)}"
    assert out["name"] == "mcp__orders__lookup_order", f"name should be 'mcp__orders__lookup_order', got {out['name']!r}"
    assert out["description"] == "Fetch one order.", "copy the description"
    assert out["input_schema"] == schema, "input_schema must be the MCP tool's input_schema, unchanged"
    bare = to_claude_tool("crm", Tool(name="ping", description=None, input_schema={"type": "object"}))
    assert bare["description"] == "", "a missing description must become an empty string, not None"
    assert bare["name"] == "mcp__crm__ping", "prefix every tool with mcp__<server>__"


run_check(1, check_ex1)

### Exercise 2 — A team `.mcp.json` without secrets

Return the dict for a project `.mcp.json` with two servers:

- `github`: remote Streamable HTTP server at `https://api.githubcopilot.com/mcp/`, sending `Authorization: Bearer <token>` where the token comes from each developer's `GITHUB_PAT` environment variable.
- `orders`: local stdio server started with `python` and the single argument `<project dir>/servers/orders_server.py`, where the project dir comes from `CLAUDE_PROJECT_DIR` (remember what a project config needs for that variable). It gets `ORDERS_DB_URL` from the developer's environment, defaulting to `postgres://localhost:5432/orders_dev`.

Set `type` explicitly on both.

In [ ]:
def build_mcp_json() -> dict:
    # TODO: return {"mcpServers": {...}}
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    cfg = build_mcp_json()
    assert isinstance(cfg, dict) and "mcpServers" in cfg, "top-level key must be 'mcpServers'"
    servers = cfg["mcpServers"]
    assert set(servers) == {"github", "orders"}, f"expected servers github and orders, got {sorted(servers)}"
    gh, orders = servers["github"], servers["orders"]
    blob = json.dumps(cfg)
    assert not re.search(r"ghp_|github_pat_", blob), "never commit a literal token"
    assert gh.get("type") in ("http", "streamable-http"), "github needs an explicit HTTP type ('http')"
    assert gh.get("url") == "https://api.githubcopilot.com/mcp/", "wrong github url"
    assert gh.get("headers", {}).get("Authorization") == "Bearer ${GITHUB_PAT}", \
        "github Authorization header must be 'Bearer ${GITHUB_PAT}'"
    assert orders.get("type") == "stdio", "orders needs type 'stdio'"
    assert orders.get("command") == "python", "orders command should be 'python'"
    args = orders.get("args", [])
    assert args == ["${CLAUDE_PROJECT_DIR:-.}/servers/orders_server.py"], \
        "args must be ['${CLAUDE_PROJECT_DIR:-.}/servers/orders_server.py'] (project configs need the :- default)"
    assert orders.get("env", {}).get("ORDERS_DB_URL") == "${ORDERS_DB_URL:-postgres://localhost:5432/orders_dev}", \
        "ORDERS_DB_URL must use ${VAR:-default} expansion"
    assert "headers" not in orders, "stdio servers do not take headers"
    json.loads(blob)


run_check(2, check_ex2)

### Exercise 3 — A structured tool error

Write `tool_error(category, message, retryable, **extra)` that returns a `CallToolResult` with:

- `is_error=True`,
- `structured_content` = `{"errorCategory": category, "isRetryable": retryable, "message": message, **extra}`,
- `content` = exactly one `TextContent` whose text is that same payload serialized with `json.dumps`.

Raise `ValueError` if `category` is not one of `transient`, `validation`, `business`, `permission`.

In [ ]:
ERROR_CATEGORIES = {"transient", "validation", "business", "permission"}


def tool_error(category: str, message: str, retryable: bool, **extra: Any) -> CallToolResult:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    r = tool_error("transient", "Orders API timed out after 5s.", True, attemptedQuery="order_id=ORD-1")
    assert isinstance(r, CallToolResult), "return a mcp.types.CallToolResult"
    assert r.is_error is True, "is_error must be True"
    expected = {"errorCategory": "transient", "isRetryable": True, "message": "Orders API timed out after 5s.",
                "attemptedQuery": "order_id=ORD-1"}
    assert r.structured_content == expected, f"structured_content should be {expected}"
    assert len(r.content) == 1 and r.content[0].type == "text", "content must be exactly one text block"
    assert json.loads(r.content[0].text) == expected, "the text block must hold the same payload as JSON"
    p = tool_error("permission", "Token expired.", False)
    assert p.structured_content["isRetryable"] is False and p.structured_content["errorCategory"] == "permission"
    try:
        tool_error("oops", "x", False)
    except ValueError:
        pass
    else:
        raise AssertionError("an unknown category must raise ValueError")


run_check(3, check_ex3)

### Exercise 4 — Empty result vs real failure

Write `make_customer_server(crm)` that returns an `MCPServer` with one tool, `search_customers(email: str)`. `crm(email)` is a plain function you receive; it returns a list of customer dicts, or raises `TimeoutError` (CRM slow) or `PermissionError` (service token expired). Required behavior:

| Situation | Result |
|---|---|
| `email` has no `@` | `is_error=True`, and the message the model sees mentions `@` (use `ToolError`) |
| CRM returns `[]` | **success** (`is_error` false) with an empty list in the payload |
| CRM returns customers | success with the customers in the payload |
| `TimeoutError` | `is_error=True`, `errorCategory` `transient`, `isRetryable` `true` |
| `PermissionError` | `is_error=True`, `errorCategory` `permission`, `isRetryable` `false` |

You may use your `tool_error` from Exercise 3. The checker connects with an in-memory `Client`, so run it with `await`.

In [ ]:
def make_customer_server(crm: Callable[[str], list[dict]]) -> MCPServer:
    # TODO: create MCPServer("CRM") with a search_customers tool
    raise NotImplementedError

In [ ]:
def _payload(r: CallToolResult) -> Any:
    if r.structured_content is not None:
        return r.structured_content
    text = "\n".join(c.text for c in r.content if c.type == "text")
    try:
        return json.loads(text)
    except ValueError:
        return text


async def check_ex4() -> None:
    def fake_crm(email: str) -> list[dict]:
        if email == "slow@example.com":
            raise TimeoutError("CRM timed out")
        if email == "expired@example.com":
            raise PermissionError("token expired")
        return [{"id": 7, "email": email}] if email == "ana@example.com" else []

    server = make_customer_server(fake_crm)
    assert isinstance(server, MCPServer), "return an MCPServer instance"
    async with Client(server) as session:
        names = [t.name for t in (await session.list_tools()).tools]
        assert "search_customers" in names, f"tool 'search_customers' not found (tools: {names})"
        bad = await session.call_tool("search_customers", {"email": "not-an-email"})
        assert bad.is_error, "an email without '@' must be an isError result"
        assert "@" in "".join(c.text for c in bad.content if c.type == "text"), \
            "the error text should tell the model the email needs '@' (raise ToolError with that message)"
        empty = await session.call_tool("search_customers", {"email": "nobody@example.com"})
        assert not empty.is_error, "no matching customer is a SUCCESS with an empty list, not an error"
        assert "[]" in json.dumps(_payload(empty)), "the empty result should contain an empty list"
        hit = await session.call_tool("search_customers", {"email": "ana@example.com"})
        assert not hit.is_error and '"id": 7' in json.dumps(_payload(hit)), "a match should return the customer"
        slow = await session.call_tool("search_customers", {"email": "slow@example.com"})
        p = _payload(slow)
        assert slow.is_error and isinstance(p, dict), "a timeout must be an isError result with a structured payload"
        assert p.get("errorCategory") == "transient" and p.get("isRetryable") is True, "timeout -> transient, retryable"
        expired = await session.call_tool("search_customers", {"email": "expired@example.com"})
        p = _payload(expired)
        assert expired.is_error and isinstance(p, dict), "an expired token must be an isError result with a payload"
        assert p.get("errorCategory") == "permission" and p.get("isRetryable") is False, \
            "expired token -> permission, not retryable"


await arun_check(4, check_ex4)

### Exercise 5 — MCP result to Claude `tool_result`

Write `to_tool_result_block(tool_use_id, result)` that converts a `CallToolResult` into a Messages API block:

- `{"type": "tool_result", "tool_use_id": ..., "content": <str>, "is_error": <bool>}`
- `content` joins the text of all **text** blocks with `"\n"` (ignore images and other block types);
- if there is no text at all but `structured_content` is set, use `json.dumps(structured_content)`;
- `is_error` mirrors the MCP flag as a real `bool`.

In [ ]:
def to_tool_result_block(tool_use_id: str, result: CallToolResult) -> dict:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    r = CallToolResult(content=[TextContent(type="text", text="line 1"),
                                ImageContent(type="image", data="aGk=", mime_type="image/png"),
                                TextContent(type="text", text="line 2")])
    b = to_tool_result_block("toolu_1", r)
    assert b.get("type") == "tool_result" and b.get("tool_use_id") == "toolu_1", "type/tool_use_id are wrong"
    assert b.get("content") == "line 1\nline 2", f"join text blocks with newlines and skip images; got {b.get('content')!r}"
    assert b.get("is_error") is False, "is_error must be the bool False for a successful result"
    s = to_tool_result_block("toolu_2", CallToolResult(content=[], structured_content={"result": []}))
    assert s["content"] == json.dumps({"result": []}), "with no text, fall back to json.dumps(structured_content)"
    e = to_tool_result_block("toolu_3", CallToolResult(content=[TextContent(type="text", text="boom")], is_error=True))
    assert e["is_error"] is True, "propagate isError to is_error"


run_check(5, check_ex5)

### Exercise 6 — The bridge loop

Write `agent_loop(llm, call_tool, tools, question, max_turns=5)` for a Messages API tool loop over MCP:

- `llm.messages.create(model=MODEL, max_tokens=..., tools=tools, messages=messages)` returns a response with `.content` (blocks with `.type`, and `.id`, `.name`, `.input` for `tool_use`, `.text` for `text`) and `.stop_reason`.
- Start with one user message (the question). After each response, append `{"role": "assistant", "content": response.content}`.
- If `stop_reason` is not `"tool_use"`, return the concatenated text of the text blocks.
- Otherwise run **every** `tool_use` block with `await call_tool(name, input)` (it returns a `CallToolResult`) and send **all** results back in **one** user message as `tool_result` blocks (use `to_tool_result_block`).
- If the model is still calling tools after `max_turns` responses, raise `RuntimeError`.

In [ ]:
async def agent_loop(llm, call_tool: Callable[[str, dict], Awaitable[CallToolResult]],
                     tools: list[dict], question: str, max_turns: int = 5) -> str:
    # TODO
    raise NotImplementedError

In [ ]:
class FakeLLM:
    """Replays scripted responses and records a snapshot of every request."""

    def __init__(self, responses):
        self.responses, self.requests = list(responses), []
        self.messages = self

    async def create(self, **kwargs):
        self.requests.append({**kwargs, "messages": list(kwargs.get("messages", []))})
        return self.responses.pop(0) if len(self.responses) > 1 else self.responses[0]


def _tool_use(id_: str, name: str, args: dict) -> NS:
    return NS(type="tool_use", id=id_, name=name, input=args)


async def check_ex6() -> None:
    calls = []

    async def fake_call_tool(name: str, args: dict) -> CallToolResult:
        calls.append((name, args))
        if args.get("order_id") == "ORD-2":
            return CallToolResult(content=[TextContent(type="text", text='{"errorCategory": "transient"}')], is_error=True)
        return CallToolResult(content=[TextContent(type="text", text='{"status": "shipped"}')])

    tools = [{"name": "lookup_order", "description": "", "input_schema": {"type": "object"}}]
    llm = FakeLLM([
        NS(stop_reason="tool_use", content=[NS(type="text", text="Checking both."),
                                            _tool_use("t1", "lookup_order", {"order_id": "ORD-1"}),
                                            _tool_use("t2", "lookup_order", {"order_id": "ORD-2"})]),
        NS(stop_reason="end_turn", content=[NS(type="text", text="ORD-1 shipped; ORD-2 lookup timed out.")]),
    ])
    answer = await agent_loop(llm, fake_call_tool, tools, "Where are ORD-1 and ORD-2?")
    assert answer == "ORD-1 shipped; ORD-2 lookup timed out.", f"return the final text, got {answer!r}"
    assert len(llm.requests) == 2, f"expected 2 model calls, got {len(llm.requests)}"
    assert llm.requests[0]["tools"] == tools, "pass the tools list to every request"
    assert calls == [("lookup_order", {"order_id": "ORD-1"}), ("lookup_order", {"order_id": "ORD-2"})], \
        "run every tool_use block, in order"
    msgs = llm.requests[1]["messages"]
    assert [m["role"] for m in msgs] == ["user", "assistant", "user"], \
        "second request must be: user question, assistant turn, ONE user message with all tool results"
    results = msgs[2]["content"]
    assert isinstance(results, list) and len(results) == 2, "send both tool_result blocks in a single user message"
    assert [r["tool_use_id"] for r in results] == ["t1", "t2"], "each tool_result must reference its tool_use id"
    assert [r["is_error"] for r in results] == [False, True], "propagate is_error from the MCP result"

    looping = FakeLLM([NS(stop_reason="tool_use", content=[_tool_use("t9", "lookup_order", {"order_id": "ORD-1"})])])
    try:
        await agent_loop(looping, fake_call_tool, tools, "loop forever", max_turns=3)
    except RuntimeError:
        pass
    else:
        raise AssertionError("raise RuntimeError when max_turns is exceeded")
    assert len(looping.requests) == 3, f"max_turns=3 should allow exactly 3 model calls, got {len(looping.requests)}"


await arun_check(6, check_ex6)

### Exercise 7 — MIME-aware resource reader

A host that injects resources into Claude's context must decode each read correctly. Write `async def read_as(session, uri)` that reads one resource and returns its first content item as a Python value:

- a binary item (`BlobResourceContents`, base64 in `.blob`) → the decoded `bytes`;
- a text item whose `mime_type` is `application/json` → the parsed JSON value (`json.loads`);
- any other text item → the `str` unchanged. Trust the label: JSON-looking text labelled `text/plain` stays a string.

In [ ]:
async def read_as(session, uri: str) -> Any:
    # TODO: await session.read_resource(uri), take .contents[0], decode by type / mime_type
    raise NotImplementedError

In [ ]:
async def check_ex7() -> None:
    srv = MCPServer("Ex7")

    @srv.resource("cfg://settings", mime_type="application/json")
    def settings() -> dict:
        return {"theme": "dark", "retries": 3}

    @srv.resource("cfg://readme", mime_type="text/markdown")
    def readme() -> str:
        return "# Readme"

    @srv.resource("cfg://raw")                     # JSON text, but labelled text/plain (the default)
    def raw() -> dict:
        return {"label": "missing"}

    @srv.resource("cfg://icon", mime_type="image/png")
    def icon() -> bytes:
        return b"\x89PNG-bytes"

    async with Client(srv) as session:
        v = await read_as(session, "cfg://settings")
        assert v == {"theme": "dark", "retries": 3}, f"application/json must be parsed into a dict, got {v!r}"
        v = await read_as(session, "cfg://readme")
        assert v == "# Readme", f"text/markdown must stay a str, got {v!r}"
        v = await read_as(session, "cfg://raw")
        assert isinstance(v, str), "text/plain must stay a str even when it looks like JSON (trust the mime_type)"
        v = await read_as(session, "cfg://icon")
        assert v == b"\x89PNG-bytes", f"a blob must be base64-decoded into the original bytes, got {v!r}"


await arun_check(7, check_ex7)

### Exercise 8 — Argument completion

A ticketing server has a prompt `triage(project)` and a resource template `tickets://{project}/{ticket_id}`. Write the completion handler `suggest(ref, argument, context)`:

- for the argument `project`: every key of `PROJECTS` that starts with the typed value, **case-insensitively**, sorted;
- for `ticket_id`: the ids of the project already chosen in `context.arguments["project"]` that start with the typed value; if no project has been chosen yet (no `context` or no `"project"` key), return `None`;
- for any other argument return `None` (the SDK turns `None` into an empty list).

Return a `Completion(values=[...])`. `argument.name` and `argument.value` hold the argument name and the typed prefix.

In [ ]:
PROJECTS = {"web": ["WEB-101", "WEB-102", "WEB-230"], "mobile": ["MOB-7", "MOB-8"], "warehouse": ["WH-1"]}

In [ ]:
async def suggest(ref: PromptReference | ResourceTemplateReference, argument: CompletionArgument,
                  context: CompletionContext | None) -> Completion | None:
    # TODO
    raise NotImplementedError

In [ ]:
async def check_ex8() -> None:
    # Call the handler directly once: inside a server, a NotImplementedError would be hidden behind a protocol error.
    await suggest(PromptReference(name="triage"), CompletionArgument(name="project", value=""), None)
    srv = MCPServer("Tickets")

    @srv.prompt()
    def triage(project: str) -> str:
        """Triage the open tickets of a project."""
        return f"Triage {project}"

    @srv.resource("tickets://{project}/{ticket_id}")
    def ticket(project: str, ticket_id: str) -> str:
        return f"{project}/{ticket_id}"

    srv.completion()(suggest)
    template = ResourceTemplateReference(uri="tickets://{project}/{ticket_id}")
    async with Client(srv) as session:
        assert session.server_capabilities.completions is not None, "registering a handler declares the capability"
        got = (await session.complete(PromptReference(name="triage"), {"name": "project", "value": "W"})).completion.values
        assert got == ["warehouse", "web"], f"project 'W' -> case-insensitive prefix match, sorted; got {got}"
        got = (await session.complete(template, {"name": "project", "value": ""})).completion.values
        assert got == ["mobile", "warehouse", "web"], f"an empty prefix lists every project, sorted; got {got}"
        got = (await session.complete(template, {"name": "ticket_id", "value": "WEB-1"},
                                      context_arguments={"project": "web"})).completion.values
        assert got == ["WEB-101", "WEB-102"], f"ticket ids come from the chosen project; got {got}"
        got = (await session.complete(template, {"name": "ticket_id", "value": ""})).completion.values
        assert got == [], f"no project chosen yet -> None (empty list); got {got}"
        got = (await session.complete(PromptReference(name="triage"), {"name": "other", "value": ""})).completion.values
        assert got == [], f"unknown argument -> None (empty list); got {got}"


await arun_check(8, check_ex8)

### Optional live bonus (live — calls the API, ~$0.01) · not graded

Wire your pieces together: the Exercise 4 server, Exercise 1 namespacing, and the Exercise 6 loop with real Claude. The `call_tool` adapter strips the `mcp__crm__` prefix before calling the MCP server.

In [ ]:
async def live_bonus() -> None:
    crm = make_customer_server(lambda email: [{"id": 7, "email": email, "plan": "pro"}] if email == "ana@example.com" else [])
    async with Client(crm) as session:
        tools = [to_claude_tool("crm", t) for t in (await session.list_tools()).tools]

        async def call_tool(name: str, args: dict) -> CallToolResult:
            return await session.call_tool(name.removeprefix("mcp__crm__"), args)

        print(await agent_loop(aclient, call_tool, tools, "Is ana@example.com a customer, and what about bob@example.com?"))


try:
    await live_bonus()
except BaseException as e:  # noqa: BLE001 - unwrap anyio's ExceptionGroup to show the real cause
    while isinstance(e, BaseExceptionGroup) and len(e.exceptions) == 1:
        e = e.exceptions[0]
    print(f"Live bonus did not run: {type(e).__name__}: {e}")

---
## Part C — Architecture scenario

**Northwind Retail** wants Claude in two places:

1. A **customer-support backend** built on the Messages API (no Claude Code), which must look up orders, check shipment status, and issue refunds (refunds above $500 need a supervisor).
2. **Claude Code** for 25 engineers, who need GitHub, Sentry, and read-only access to the analytics Postgres warehouse (about 60 tables).

Existing systems: an internal orders/refunds REST API, GitHub, Sentry, Postgres. The security team insists that no credential is ever committed and that refund limits cannot be bypassed by prompt injection.

Design the MCP layer. Cover: which servers you build vs reuse; transport for each; how Claude Code is configured (scopes, files, secrets); which primitives you expose (and why); the error contract; how you limit what each agent can call; and how the support backend reaches its tools.

*Your answer here (double-click to edit).*

<details><summary>Rubric: what a strong answer contains (open after writing yours)</summary>

- **Build vs reuse:** reuse the official/maintained GitHub and Sentry MCP servers; build one custom **orders** server with narrow, purpose-built tools (`lookup_order`, `get_shipment_status`, `request_refund`), not a generic "call any endpoint" tool. For Postgres, reuse a read-only server or build a narrow one; in either case use a read-only database role.
- **Transport:** the orders server is deployed once as **Streamable HTTP** behind TLS with OAuth/bearer auth, because the support backend and possibly many engineers share it and the Messages API connector can only reach remote HTTP servers. GitHub/Sentry are remote HTTP servers. Local stdio only for dev-machine tools. No new HTTP+SSE.
- **Claude Code config:** a committed project **`.mcp.json`** with `type` set on every entry and secrets via `${GITHUB_PAT}`, `${SENTRY_TOKEN}`, `${WAREHOUSE_DB_URL:-...}` expansion; the README lists required variables; `.mcp.json` changes are code-reviewed; personal experiments at local scope. Mentions that each developer approves project servers once.
- **Primitives:** the warehouse schema and a table catalog as **resources** (avoids 4–6 exploratory calls on 60 tables); the query capability as a **tool**; team workflows (e.g. `/triage-incident`) as **prompts**. Notes the connector is tools only, so anything the support backend needs must be a tool.
- **Error contract:** `isError: true` with `errorCategory` (transient/validation/business/permission), `isRetryable`, message and attempted query; empty results are successes; refund-over-limit is a non-retryable `business` error with an escalation hint; bounded retries for transient errors only.
- **Least privilege:** each agent sees only what its role needs (`allowed_tools` / connector `mcp_toolset` allowlist with `default_config.enabled=false`); separate read-only vs write tools; tool annotations treated as hints, not security.
- **Enforcement:** the $500 limit is enforced **in the orders server** (and/or a host hook), never only in the prompt or a tool description; auth tokens scoped per service, with no Claude/cloud credentials reused for third-party servers.
- **Support backend path:** Messages API **MCP connector** (beta, `mcp_servers` + `mcp_toolset`) pointing at the HTTP orders server with an authorization token, **or** a backend-side MCP `Client` bridge when it needs full loop control, local servers, or non-tool primitives; justifies the choice.

</details>

---
## Scorecard

In [ ]:
quiz = RESULTS["quiz"] if RESULTS["quiz"] is not None else 0.0
passed = sum(1 for n in range(1, N_EXERCISES + 1) if RESULTS["exercises"].get(n))
ex_score = passed / N_EXERCISES
overall = (quiz + ex_score) / 2

print(f"Quiz:       {quiz:.0%}" + ("   (run the grader in Part A)" if RESULTS["quiz"] is None else ""))
print(f"Exercises:  {passed}/{N_EXERCISES} = {ex_score:.0%}")
for n in range(1, N_EXERCISES + 1):
    print(f"   Exercise {n}: {'✓' if RESULTS['exercises'].get(n) else '✗'}")
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS" if overall >= 0.72 else "RETRY: revisit the README sections behind the ✗ items, then try again.")

## Recap

- Pick the primitive by who controls it: tools (model), resources (host/user), prompts (user).
- stdio for local single-user servers (stdout is the protocol channel); Streamable HTTP for anything shared or reached by the Messages API connector.
- Team servers go in a committed `.mcp.json` with `${VAR}` / `${VAR:-default}` expansion; local > project > user, no merging.
- Tool failures: `isError` + `errorCategory` + `isRetryable` + message, via `ToolError` or a returned `CallToolResult`; empty results are successes.
- Resources: declare `mime_type` and decode reads by it (blob → bytes, `application/json` → parsed, other text → str). One completion handler serves prompt arguments and template parameters; `None` means no suggestions.
- stdio servers launched by the Python SDK see only a safe subset of the host environment: pass secrets in `env`.
- A bridge is small: translate tool definitions, run every `tool_use`, return all `tool_result` blocks in one message, map `isError` to `is_error`, cap the turns.

Next chapter: [Chapter 5 — Claude Code](../05-claude-code/README.md)

---
## Solutions (spoilers)

<details><summary>Quiz answers and explanations</summary>

- **Q1 — C.** A user-picked, parameterized workflow template is a **prompt**; Claude Code lists MCP prompts as slash commands (`/mcp__<server>__<prompt>`). A tool is model-controlled; a resource is read-only context; annotations change nothing about who triggers a workflow.
- **Q2 — B.** Shared, multi-user, and connector-reachable means **Streamable HTTP** with auth. The connector cannot launch or reach stdio processes (A, D), and HTTP+SSE is deprecated (C), so do not build new servers on it.
- **Q3 — D.** In a stdio server, stdout *is* the JSON-RPC channel; stray prints corrupt it. Log to stderr. The other options do not touch the cause.
- **Q4 — B.** Environment variable expansion keeps one reviewed config without secrets; the leaked token must be revoked. A leaks secrets into git history, C loses the shared definition, D is needless engineering.
- **Q5 — A.** Precedence is local > project > user, and the winning entry is used **whole**; fields are not merged.
- **Q6 — C.** "No match" is a valid answer, not a failure. Flagging it as an error triggers retries and wrong "system is down" messages. B hides it from the model; D hides real outages.
- **Q7 — D.** In SDK v2 only `ToolError` (or a returned `CallToolResult`) puts your message in front of the model; other exceptions are sanitized to `Error executing tool <name>`. `MCPError` is a protocol error the model never sees.
- **Q8 — A.** The Messages API MCP connector supports **tools only**. The `mcp-client-2026-09-15` header adds tool-list pinning, not resources or prompts.
- **Q9 — C.** Selection reliability falls as tool count grows; give the role only what it needs. A and D add cost without fixing the cause; B destroys the descriptions the model relies on.
- **Q10 — B.** Annotations are hints (and the spec says clients must treat them as untrusted unless the server is trusted). Hard limits belong in deterministic code: the server or a hook.
- **Q11 — D.** `str` becomes text, `bytes` a base64 blob, anything else JSON text; `mime_type` is only what you declare, and it defaults to `text/plain`. A changes the payload into binary for no reason, B moves read-only context into the model-controlled primitive, C adds a parameter but leaves the label wrong.
- **Q12 — A.** The Python SDK's stdio client starts the server with a safe default list (`HOME`, `LOGNAME`, `PATH`, `SHELL`, `TERM`, `USER` on POSIX) merged with `env`, so secrets must be passed on purpose. That is also what the `env` field of `.mcp.json` and of Agent SDK server configs is for. B commits a secret, C is false, and D corrupts the protocol channel.

</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
def to_claude_tool(server_name: str, tool: Tool) -> dict:
    return {
        "name": f"mcp__{server_name}__{tool.name}",
        "description": tool.description or "",
        "input_schema": tool.input_schema,
    }
```

</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
def build_mcp_json() -> dict:
    return {
        "mcpServers": {
            "github": {
                "type": "http",
                "url": "https://api.githubcopilot.com/mcp/",
                "headers": {"Authorization": "Bearer ${GITHUB_PAT}"},
            },
            "orders": {
                "type": "stdio",
                "command": "python",
                "args": ["${CLAUDE_PROJECT_DIR:-.}/servers/orders_server.py"],
                "env": {"ORDERS_DB_URL": "${ORDERS_DB_URL:-postgres://localhost:5432/orders_dev}"},
            },
        }
    }
```

</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
ERROR_CATEGORIES = {"transient", "validation", "business", "permission"}


def tool_error(category: str, message: str, retryable: bool, **extra: Any) -> CallToolResult:
    if category not in ERROR_CATEGORIES:
        raise ValueError(f"unknown error category {category!r}")
    payload = {"errorCategory": category, "isRetryable": retryable, "message": message, **extra}
    return CallToolResult(
        content=[TextContent(type="text", text=json.dumps(payload))],
        structured_content=payload,
        is_error=True,
    )
```

</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
def make_customer_server(crm: Callable[[str], list[dict]]) -> MCPServer:
    server = MCPServer("CRM")

    @server.tool()
    def search_customers(email: str) -> CallToolResult:
        """Find customers by exact email. An empty list means no customer has that email."""
        if "@" not in email:
            raise ToolError("email must contain '@'.")
        try:
            rows = crm(email)
        except TimeoutError:
            return tool_error("transient", "CRM timed out.", True, attemptedQuery=f"email={email}")
        except PermissionError:
            return tool_error("permission", "CRM service token expired; escalate to an admin.", False)
        payload = {"customers": rows}
        return CallToolResult(content=[TextContent(type="text", text=json.dumps(payload))],
                              structured_content=payload)

    return server
```

</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def to_tool_result_block(tool_use_id: str, result: CallToolResult) -> dict:
    text = "\n".join(c.text for c in result.content if c.type == "text")
    if not text and result.structured_content is not None:
        text = json.dumps(result.structured_content)
    return {"type": "tool_result", "tool_use_id": tool_use_id, "content": text, "is_error": bool(result.is_error)}
```

</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
async def agent_loop(llm, call_tool: Callable[[str, dict], Awaitable[CallToolResult]],
                     tools: list[dict], question: str, max_turns: int = 5) -> str:
    messages: list[dict] = [{"role": "user", "content": question}]
    for _ in range(max_turns):
        resp = await llm.messages.create(model=MODEL, max_tokens=600, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for block in resp.content:
            if block.type == "tool_use":
                result = await call_tool(block.name, block.input)
                results.append(to_tool_result_block(block.id, result))
        messages.append({"role": "user", "content": results})
    raise RuntimeError(f"agent did not finish within {max_turns} turns")
```

</details>

<details><summary>Exercise 7 — reference solution</summary>

```python
async def read_as(session, uri: str) -> Any:
    item = (await session.read_resource(uri)).contents[0]
    if isinstance(item, BlobResourceContents):
        return base64.b64decode(item.blob)
    if item.mime_type == "application/json":
        return json.loads(item.text)
    return item.text
```

</details>

<details><summary>Exercise 8 — reference solution</summary>

```python
async def suggest(ref: PromptReference | ResourceTemplateReference, argument: CompletionArgument,
                  context: CompletionContext | None) -> Completion | None:
    typed = argument.value
    if argument.name == "project":
        return Completion(values=sorted(p for p in PROJECTS if p.lower().startswith(typed.lower())))
    if argument.name == "ticket_id":
        project = (context.arguments or {}).get("project") if context else None
        if project is None:
            return None
        return Completion(values=[t for t in PROJECTS.get(project, []) if t.startswith(typed)])
    return None
```

</details>